<a href="https://colab.research.google.com/github/shaiksameer786/Gen-AI-experiments/blob/main/Gen_AI_exp_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install all required packages
!pip install -q transformers datasets evaluate sentence-transformers rouge_score

import torch
from transformers import pipeline, set_seed
import evaluate
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# Set seed for reproducibility
set_seed(42)

# Use GPU if available
device = 0 if torch.cuda.is_available() else -1

# Load text generation pipeline
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

# Fix GPT-2 padding issue
generator.tokenizer.pad_token = generator.tokenizer.eos_token

print("Model loaded successfully!\n")

# Prompts
prompts = {
    "Basic Prompt":
        "Explain machine learning.",

    "Instruction Prompt":
        "Explain machine learning in simple and clear terms.",

    "Role-Based Prompt":
        "You are a professor. Explain machine learning in a detailed academic way.",

    "Few-Shot Prompt":
        """Example:
AI is the ability of machines to mimic human intelligence.

Now explain:
Machine learning is"""
}

generated_outputs = {}

print("=== GENERATED OUTPUTS ===\n")

# Generate outputs
for key, prompt in prompts.items():
    result = generator(
        prompt,
        max_new_tokens=120,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    text = result[0]['generated_text']
    generated_outputs[key] = text

    print(f"\n--- {key} ---")
    print(text)
    print("="*100)

# Reference text
reference = ["Machine learning is a subset of artificial intelligence that allows systems to learn from data and improve automatically."]

# Load evaluation metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Load semantic similarity model
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

reference_embedding = semantic_model.encode(reference[0], convert_to_tensor=True)

results = []

print("\n\n=== EVALUATION RESULTS ===\n")

# Evaluate each output
for key, output in generated_outputs.items():

    prediction = [output]

    # BLEU expects tokenized references
    bleu_score = bleu.compute(
        predictions=prediction,
        references=[[reference[0]]]
    )["bleu"]

    # ROUGE
    rouge_score = rouge.compute(
        predictions=prediction,
        references=reference
    )["rougeL"]

    # Semantic Similarity
    output_embedding = semantic_model.encode(output, convert_to_tensor=True)
    similarity = util.cos_sim(reference_embedding, output_embedding).item()

    results.append([key, bleu_score, rouge_score, similarity])

    print(f"{key}")
    print(f"BLEU Score            : {bleu_score:.4f}")
    print(f"ROUGE-L Score         : {rouge_score:.4f}")
    print(f"Semantic Similarity   : {similarity:.4f}")
    print("-"*60)

# Create DataFrame
df = pd.DataFrame(results, columns=[
    "Prompt Type",
    "BLEU Score",
    "ROUGE-L",
    "Semantic Similarity"
])

print("\n\n=== FINAL COMPARISON TABLE ===\n")
print(df)

# Best prompt
best_prompt = df.sort_values(by="Semantic Similarity", ascending=False).iloc[0]

print("\nBest Performing Prompt Based on Semantic Similarity:")
print(best_prompt)

  Preparing metadata (setup.py) ... done


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_p', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded successfully!

=== GENERATED OUTPUTS ===



Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]



=== EVALUATION RESULTS ===

Basic Prompt
BLEU Score            : 0.0000
ROUGE-L Score         : 0.1167
Semantic Similarity   : 0.7047
------------------------------------------------------------
Instruction Prompt
BLEU Score            : 0.0000
ROUGE-L Score         : 0.1538
Semantic Similarity   : 0.7322
------------------------------------------------------------
Role-Based Prompt
BLEU Score            : 0.0000
ROUGE-L Score         : 0.0602
Semantic Similarity   : 0.5037
------------------------------------------------------------
Few-Shot Prompt
BLEU Score            : 0.0000
ROUGE-L Score         : 0.1085
Semantic Similarity   : 0.6104
------------------------------------------------------------


=== FINAL COMPARISON TABLE ===

          Prompt Type  BLEU Score   ROUGE-L  Semantic Similarity
0        Basic Prompt         0.0  0.116667             0.704676
1  Instruction Prompt         0.0  0.153846             0.732247
2   Role-Based Prompt         0.0  0.060150             0.5